In [2]:
import sys
import requests
import json
import vlc
import re
import random
from IPython.display import Audio

def load_json_database(url):
    records = []
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad status codes
        for line in response.iter_lines(decode_unicode=True):
            if line:
                try:
                    record = json.loads(line)
                    records.append(record)
                except json.JSONDecodeError as e:
                    print(f"Error parsing JSON: {e}")
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from URL: {e}")
    return records

def search_in_json_database(database, search_word, region):
    for record in database:
        # Check whether the word field matches
        if record.get('word') == search_word:
            # When a match is found, get US pronunciation data
            pos_items = record.get('pos_items', [])
            for pos_item in pos_items:
                pronunciations = pos_item.get('pronunciations', [])
                for pronunciation in pronunciations:
                    if pronunciation.get('region') == region:
                        # Return data when US pronunciation is found
                        return {
                            'pronunciation': pronunciation.get('pronunciation'),
                            'audio': pronunciation.get('audio')
                        }
    # Return not exist when no word field matches
    return 'not exist'

url = "https://raw.githubusercontent.com/zelic91/camdict/main/cam_dict.refined.json"

json_database = load_json_database(url)


In [9]:
text ="""
Như đã nói, nguyên âm trong âm tiết không nhấn có thể bị giảm thành schwa `ə`. Trong lời nói tự nhiên, ngay cả schwa `ə` cũng có thể giảm thêm thành âm rất nhẹ `ɤ`. Chẳng hạn, từ thông dụng reasonable, `/ˈriːzənəbəl/`, thường được nghe là `/ˈriːzɤnəbəl/`.
"""

words = ""
phonetics_not_exist = ''
lines = text.split("\n")
for line in lines:
    line = line.replace(" `", " <span class=\"pho\">")
    line = line.replace("`", "</span>")
    if '*' in line:
        line = line.replace('"', "**")
        examples = line.split("-")[1].split(",")
        examples = [x.strip() for x in examples]
        line = line.replace(" - ", "\n")
        # print(examples)
        # wrap examples in span,
        for e in examples:
            # join e in words with ','
            words += e + ","
            entry_us = search_in_json_database(json_database, e, 'us')
            if entry_us == 'not exist':
                phonetics = entry_us
                phonetics_not_exist += f'{e},'
            else:
                phonetics = entry_us['pronunciation']    
            wrapped_e = f'\t- {e} <span class="pho alt">{phonetics}</span> <span class="speak-word-inline" data-audio-us-male="/audios/us/{e}-us-male.mp3" data-audio-us-female="/audios/us/{e}-us-female.mp3"></span>\n'
            line = line.replace(e, wrapped_e).replace(',', '').strip()
            

    print(line)
print(f'phonetics_not_exist: {phonetics_not_exist}')
print('\n'+words)


Như đã nói, nguyên âm trong âm tiết không nhấn có thể bị giảm thành schwa <span class="pho">ə</span>. Trong lời nói tự nhiên, ngay cả schwa <span class="pho">ə</span> cũng có thể giảm thêm thành âm rất nhẹ <span class="pho">ɤ</span>. Chẳng hạn, từ thông dụng reasonable, </span>/ˈriːzənəbəl/</span>, thường được nghe là </span>/ˈriːzɤnəbəl/</span>.

phonetics_not_exist: 




In [11]:
import asyncio
import edge_tts
import pygame
text = words.rstrip(",")
Wordlist = text.split(",")
# Wordlist = ['reasonable']

print(Wordlist)
for w in Wordlist:
  # for VOICE in ['en-US-GuyNeural', 'en-US-JennyNeural', 'en-GB-RyanNeural', 'en-GB-SoniaNeural']:
  for VOICE in ['en-US-GuyNeural', 'en-US-MichelleNeural']:
    w = w.strip()
    # OUTPUT_FILE = f"{w}-{VOICE.replace('EricNeural', 'Guy-Male').replace('JennyNeural', 'Jenny-Female').replace('RyanNeural', 'Ryan-Male').replace('SoniaNeural', 'Sonia-Female').lower()}.mp3"
    OUTPUT_FILE = f"{w}-{VOICE.replace('GuyNeural', 'Male').replace('MichelleNeural', 'Female').replace('en-', '').lower()}.mp3"
    communicate = edge_tts.Communicate(w, VOICE)
    await communicate.save(OUTPUT_FILE) 
  print(w)
print("Files created!") 

['reasonable']
reasonable
Files created!
